# Stadt Zürich - Geo-Map 


Wir haben alles was wir brauchen:    

gtfs_tram_shapes.parquet → Linienverläufe als Koordinaten    
gtfs_tram_stops.parquet → Haltestellen mit Lat/Lon     
gtfs_tram_routes.parquet → Liniennamen und Farben     

Öffne vbz-geo-map.ipynb und fang mit dem Laden an:     

In [2]:
import pandas as pd
from pathlib import Path

GTFS_DIR = Path("../../data/interim/vbz/gtfs/")

routes = pd.read_parquet(GTFS_DIR / "gtfs_tram_routes.parquet")
stops  = pd.read_parquet(GTFS_DIR / "gtfs_tram_stops.parquet")
shapes = pd.read_parquet(GTFS_DIR / "gtfs_tram_shapes.parquet")

print(f"Routen: {len(routes)}")
print(f"Stops:  {len(stops)}")
print(f"Shapes: {len(shapes):,}")

print("\nRouten Spalten:", routes.columns.tolist())
print("Stops Spalten: ", stops.columns.tolist())
print("Shapes Spalten:", shapes.columns.tolist())

Routen: 49
Stops:  7202
Shapes: 535,930

Routen Spalten: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'year']
Stops Spalten:  ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'year']
Shapes Spalten: ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled', 'year']


In [3]:
# Linienfarben anschauen
print(routes[["route_short_name", "route_color", "route_text_color", "year"]]
      .sort_values(["route_short_name", "year"])
      .to_string())

   route_short_name route_color route_text_color  year
0                10      E12472           FFFFFF  2023
16               10      E12472           FFFFFF  2024
32               10      E12472           FFFFFF  2025
1                11      00892F           FFFFFF  2023
17               11      00892F           FFFFFF  2024
33               11      00892F           FFFFFF  2025
2                12      92D6E3           000000  2023
18               12      92D6E3           000000  2024
34               12      92D6E3           000000  2025
3                13      FFCC00           000000  2023
19               13      FFCC00           000000  2024
35               13      FFCC00           000000  2025
4                14      008DC5           FFFFFF  2023
20               14      008DC5           FFFFFF  2024
36               14      008DC5           FFFFFF  2025
5                15      E20A16           FFFFFF  2023
21               15      E20A16           FFFFFF  2024
37        

In [9]:
# ── 2. Heatmap Style ──────────────────────────────────────────────────────
import plotly.express as px

# Mehr Punkte für Heatmap-Effekt
stops_heat = stops_2024.sample(200).copy()
stops_heat["delay"] = np.random.exponential(scale=120, size=200)

fig2 = px.density_mapbox(
    stops_heat,
    lat="stop_lat",
    lon="stop_lon",
    z="delay",
    radius=25,
    center=dict(lat=47.378, lon=8.540),
    zoom=12,
    mapbox_style="carto-darkmatter",
    color_continuous_scale="Inferno",
    title="Test: Verspätungs-Heatmap (synthetische Daten)"
)

fig2.update_layout(height=600, margin=dict(l=0, r=0, t=30, b=0))
fig2.show()

/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_5013/2566586281.py:8: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2 = px.density_mapbox(


In [12]:
# Haltestellen auswählen (erste Übereinstimmung)
h_para  = stops_2024[stops_2024["stop_name"] == "Zürich, Paradeplatz"].iloc[0]
h_belle = stops_2024[stops_2024["stop_name"] == "Zürich, Bellevue"].iloc[0]
h_stad  = stops_2024[stops_2024["stop_name"] == "Zürich, Bahnhof Stadelhofen"].iloc[0]

# Shape der Linie 11 laden (fährt Paradeplatz -> Bellevue -> Stadelhofen)
route_11 = routes_2024[routes_2024["route_short_name"] == "11"]["route_id"]
shape_ids_11 = trips_2024[trips_2024["route_id"].isin(route_11)]["shape_id"].unique()

# Längsten Shape nehmen
best_shape = max(shape_ids_11, key=lambda s: len(shapes_2024[shapes_2024["shape_id"] == s]))
shape_pts = (
    shapes_2024[shapes_2024["shape_id"] == best_shape]
    .sort_values("shape_pt_sequence")
)

# Nächsten Shape-Punkt zu jeder Haltestelle finden
from scipy.spatial import cKDTree

tree = cKDTree(shape_pts[["shape_pt_lat", "shape_pt_lon"]].values)

def nearest_idx(halt):
    _, idx = tree.query([halt["stop_lat"], halt["stop_lon"]])
    return shape_pts.iloc[idx]["shape_pt_sequence"]

idx_para  = nearest_idx(h_para)
idx_belle = nearest_idx(h_belle)
idx_stad  = nearest_idx(h_stad)

print(f"Paradeplatz  → Shape-Index: {idx_para}")
print(f"Bellevue     → Shape-Index: {idx_belle}")
print(f"Stadelhofen  → Shape-Index: {idx_stad}")

# Segmente ausschneiden
def get_segment(from_idx, to_idx):
    lo, hi = sorted([from_idx, to_idx])
    return shape_pts[
        (shape_pts["shape_pt_sequence"] >= lo) &
        (shape_pts["shape_pt_sequence"] <= hi)
    ]

seg1 = get_segment(idx_para, idx_belle)
seg2 = get_segment(idx_belle, idx_stad)

# Karte
fig3 = go.Figure()

for seg, label, delay, farbe in [
    (seg1, "Paradeplatz → Bellevue",     4.2, "#E53935"),
    (seg2, "Bellevue → Stadelhofen",     1.1, "#43A047"),
]:
    fig3.add_trace(go.Scattermapbox(
        lat=seg["shape_pt_lat"].tolist(),
        lon=seg["shape_pt_lon"].tolist(),
        mode="lines",
        line=dict(color=farbe, width=6),
        name=f"{label} ({delay} Min)",
        hovertemplate=f"{label}<br>Verspätung: {delay} Min<extra></extra>"
    ))

# Haltestellen als Punkte
for h, name in [(h_para, "Paradeplatz"), (h_belle, "Bellevue"), (h_stad, "Stadelhofen")]:
    fig3.add_trace(go.Scattermapbox(
        lat=[h["stop_lat"]],
        lon=[h["stop_lon"]],
        mode="markers",
        marker=dict(size=12, color="white"),
        name=name,
        hovertemplate=f"{name}<extra></extra>"
    ))

fig3.update_layout(
    mapbox=dict(
        style="carto-darkmatter",
        center=dict(lat=47.379, lon=8.542),
        zoom=14
    ),
    margin=dict(l=0, r=0, t=30, b=0),
    height=600,
    title="Test: Verspätung auf Streckensegmenten — Linie 11 (synthetische Daten)"
)

fig3.show()

Paradeplatz  → Shape-Index: 372
Bellevue     → Shape-Index: 409
Stadelhofen  → Shape-Index: 429


/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_5013/3219478445.py:52: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig3.add_trace(go.Scattermapbox(
/var/folders/jh/b553h44j08x_jr8xwh9jbc5r0000gn/T/ipykernel_5013/3219478445.py:63: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig3.add_trace(go.Scattermapbox(
